# einops-rearrange-flatten composite — cx1: flatten last two axes via grouped-axis rearrange

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `einops-rearrange`, `einops-rearrange-flatten`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "einops-rearrange-flatten"
DD_ATOM_IDS = ["einops-rearrange", "einops-rearrange-flatten"]
DD_SUBTOPICS = ["Einops: Rearrange", "Einops: Rearrange-as-flatten"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

`einops-rearrange` is the general pattern syntax `'src -> dst'`. `einops-rearrange-flatten` is the *grouped-axis* sub-pattern where you wrap names in parentheses to collapse them — `'b c h w -> b c (h w)'` flattens the spatial axes while preserving batch and channel.

**The composition.** The grouped-axis flatten *is* a rearrange — there is no separate `flatten` op. The parens-grouping is the only thing that distinguishes a flatten-rearrange from an identity-rearrange. In ARENA code you reach for `'b c h w -> b c (h w)'` constantly: attention needs `(B, C, HW)` tokens, pooling needs `(B, C, HW)` to reduce, etc.

**Inner-loop order matters.** `(h w)` flattens with `w` as the inner-fastest axis (matches row-major). `(w h)` would transpose first, then flatten — different bytes.

### Composite Exercise — flatten last two axes via grouped-axis rearrange

**Atoms exercised together**: `einops-rearrange`, `einops-rearrange-flatten`

Build `cx1_flatten_grouped(x)` that takes a 4-D feature map `x` of shape `(B, C, H, W)` and returns a 3-D tensor of shape `(B, C, H*W)` using a SINGLE `rearrange` call with a grouped-axis pattern.

Constraints:
- Must use `einops.rearrange` (not `.view`, not `.reshape`, not `.flatten`).
- Pattern must group `h` and `w` into `(h w)` — in that order, so the byte layout matches `x.reshape(B, C, H*W)`.
- Batch and channel axes must remain in positions 0 and 1.

In [ ]:
def cx1_flatten_grouped(x):
    return rearrange(x, 'b c h w -> b c (h w)')


<details><summary>Show solution — cx1</summary>

```python
def cx1_flatten_grouped(x):
    return rearrange(x, 'b c h w -> b c (h w)')
```

Both atoms live in one expression: `rearrange(...)` is the rearrange atom, and the `(h w)` grouped-axis pattern is the flatten atom. Drop the parens and you get a shape error (`b c h w -> b c h w` is a different output). Reverse the order to `(w h)` and the byte layout differs from `x.reshape(B, C, H*W)` — test (a) catches it.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx1',
        'subtopics': ["Einops: Rearrange", "Einops: Rearrange-as-flatten"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()